AIMS 5702 Artificial Intelligence in Practice (Fall 2026-27)

**AIMS 5702 Assignment 3**

Please complete this Colab notebook and:

* Download it as **both** an .ipynb file and a .pdf file (for the .ipynb file, click File -> Download -> Download .ipynb).
* Submit **both** the .pdf and .ipynb files to Blackboard.

**Accelerator:** Although this training can run on a CPU, a GPU will be much faster, so we recommend using a T4.

**Submission deadline**: 29th October, 2026 (Thursday)

First, let us import all the libraries needed for this assignment.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import scipy.signal
import math
import time
from typing import List, Union, Dict, Tuple

Problem A. Backpropagation
==

In this problem, let us try to implement backpropagation ourselves.

Requirements:

* You cannot use any PyTorch backward functions.
* You cannot use any automatic differentiation libraries.
* You can use PyTorch forward functions or NumPy utility functions, unless we explicitly state that certain functions cannot be used.

## Utility functions for verifying the gradient results

First, let us create a base class from which your classes will inherit. Do not change it.

In [2]:
#@title Base class for all your implementations

class CustomizedLayer(object):

  def set_params(self, param_dict: Dict[str, np.ndarray]) -> None:
    """
    Set the parameters of the layer, such as the weight and bias of a linear layer.

    Args:
      param_dict: A dictionary of parameters.
    """
    raise NotImplementedError

  def forward(self, inputs: List[np.ndarray]) -> np.ndarray:
    """
    Compute the forward pass of the layer.

    Args:
      inputs: A list of input tensors.

    Returns:
      The output of the layer.
    """
    raise NotImplementedError

  def input_gradients(
      self,
      inputs: List[np.ndarray],
      outputs: np.ndarray,
      output_gradient: np.ndarray
  ) -> Union[np.ndarray, List[np.ndarray]]:
    """
    Calculate the input gradients of the layer.
    Args:
      inputs: A list of input tensors.
      outputs: The output of the layer.
      output_gradient: The gradient of the output.
    """
    raise NotImplementedError

  def param_gradients(
      self,
      inputs: List[np.ndarray],
      outputs: np.ndarray,
      output_gradient: np.ndarray
  ) -> Dict[str, np.ndarray]:
    """
    Calculate the parameter gradients of the layer.
    Args:
      inputs: A list of input tensors.
      outputs: The output of the layer.
      output_gradient: The gradient of the output.
    """
    raise NotImplementedError

Then, define a few utility functions for testing.

In [3]:
#@title Utility functions

def verify_output(
    actual: Union[np.ndarray, List[np.ndarray]],
    reference: Union[np.ndarray, List[np.ndarray]],
    name: Union[str, List[str]],
    atol: float = 1e-6,
    rtol: float = 1e-6
) -> bool:
  """
  Verify whether the actual output is close to the reference output.

  Args:
    actual: The actual output.
    reference: The reference output.
    atol: The absolute tolerance.
    rtol: The relative tolerance.
  """
  if isinstance(actual, np.ndarray):
    close = (isinstance(reference, np.ndarray)
             and actual.shape == reference.shape
             and np.allclose(actual, reference, atol=atol, rtol=rtol))
    if not close:
      print(f"{name} is not close to the reference output.")
    else:
      print(f"{name} is close to the reference output.")
  elif isinstance(actual, list):
    if not isinstance(reference, list) or len(actual) != len(reference):
      print(f"{name} has a different number of outputs from the reference.")
      return False
    close = all(verify_output(a, b, name[i], atol=atol, rtol=rtol)
                for i, (a, b) in enumerate(zip(actual, reference)))
  else:
    print(f"{name} is not an array or a list of arrays.")
    return False
  return close

def compare_with_actual_layer(
    official_layer: nn.Module,
    customized_layer: CustomizedLayer,
    input_np: List[np.ndarray],
    output_grad_np: np.ndarray,
    parameter_names: List[str],
    param_np: List[np.ndarray],
    a_tol: float = 1e-6,
    r_tol: float = 1e-6,
) -> bool:
  """
  Compare the forward and backward passes of your customized layer with those of the official layer.

  Args:
    official_layer: The official layer.
    customized_layer: The customized layer.
    input_np: The input NumPy arrays.
    output_grad_np: The output gradient NumPy array.
    parameter_names: The names of the parameters.
    param_np: The NumPy arrays of the parameters.
    a_tol: The absolute tolerance.
    r_tol: The relative tolerance.

  Returns:
    True if the output of the customized layer is close to the output of
    the official layer, False otherwise.
  """
  success = True
  input_torch = [
      torch.tensor(x, requires_grad=True) for x in input_np
  ]
  output_grad_torch = torch.tensor(output_grad_np, requires_grad=True)
  for i, name in enumerate(parameter_names):
    setattr(official_layer, name,
            nn.Parameter(torch.tensor(param_np[i], requires_grad=True)))

  # Run the forward and backward passes on the official layer.
  output_official_torch = official_layer(*input_torch)
  output_official_torch.backward(output_grad_torch)
  output_official_np = output_official_torch.detach().numpy()

  # Check output
  output_customized_np = customized_layer.forward(input_np)
  success = verify_output(output_customized_np, output_official_np,
                          "Output", atol=a_tol, rtol=r_tol)
  if not success:
    return success

  # Check input gradient
  input_grad_customized = customized_layer.input_gradients(
      input_np, output_official_np, output_grad_np)
  input_grad_official = [x.grad.numpy() for x in input_torch]
  success = verify_output(
      input_grad_customized,
      input_grad_official,
      [f'Input Gradient {i}' for i in range(len(input_grad_official))],
      atol=a_tol, rtol=r_tol
  )
  if not success:
    return success

  # Check parameter gradient
  parameter_grad_customized = customized_layer.param_gradients(
      input_np, output_official_np, output_grad_np)
  for name in parameter_names:
    success = verify_output(parameter_grad_customized[name],
                            getattr(official_layer, name).grad.numpy(),
                            f"Parameter Gradient {name}", atol=a_tol, rtol=r_tol)
    if not success:
      return success

  print('All tests passed')
  return success

## Question A.1 Back-propagation of MUL


Complete the implementation of the class below, then run the test below.

* We have provided the implementation of the forward pass.
* Complete the backward pass in `input_gradients`.
* Note that multiplication does not have parameters, so `param_gradients` returns an empty dictionary.

Hint: For two inputs with the same shape, apply the chain rule to the element-wise product `x * y`.


In [ ]:
#@title CustomizedMul

class CustomizedMul(CustomizedLayer):
  def __init__(self):
    pass

  def set_params(self, param_dict: Dict[str, np.ndarray]) -> None:
    pass

  def forward(self, inputs: List[np.ndarray]) -> np.ndarray:
    x, y = inputs
    return x * y

  def input_gradients(
    self,
    inputs: List[np.ndarray],
    outputs: np.ndarray,
    output_gradient: np.ndarray
  ) -> Union[np.ndarray, List[np.ndarray]]:
    #####################
    # Your code goes here
    #####################
    x, y = inputs
    return [output_gradient * y, output_gradient * x]

  def param_gradients(
      self,
      inputs: List[np.ndarray],
      outputs: np.ndarray,
      output_gradient: np.ndarray
  ) -> Dict[str, np.ndarray]:
    return {}

Run the test below. Do not change it.

In [5]:
#@title Run test

def verify_mul():
  for width, height in [(3, 3), (10, 1), (100, 300)]:
    print(f'=== Testing with width={width} and height={height} ===')
    if width == 1:
      x1 = np.random.randn(height)
      x2 = np.random.randn(height)
    else:
      x1 = np.random.randn(height, width)
      x2 = np.random.randn(height, width)
    output_gradient = np.random.randn(*x1.shape)
    customized_mul = CustomizedMul()
    customized_mul.set_params({})
    class TorchMul(nn.Module):
      def __init__(self):
        super(TorchMul, self).__init__()
      def forward(self, x1, x2):
        return x1 * x2
    torch_mul = TorchMul()

    compare_with_actual_layer(
        torch_mul,
        customized_mul,
        [x1, x2],
        output_gradient,
        param_np=[],
        parameter_names=[],
    )

verify_mul()

=== Testing with width=3 and height=3 ===
Output is close to the reference output.
Input Gradient 0 is close to the reference output.
Input Gradient 1 is close to the reference output.
All tests passed
=== Testing with width=10 and height=1 ===
Output is close to the reference output.
Input Gradient 0 is close to the reference output.
Input Gradient 1 is close to the reference output.
All tests passed
=== Testing with width=100 and height=300 ===
Output is close to the reference output.
Input Gradient 0 is close to the reference output.
Input Gradient 1 is close to the reference output.
All tests passed


## Question A.2 Back-propagation of ReLU(x * y)


Similarly, complete the implementation of the class below, then run the test below.

* We have provided the implementation of the forward pass.
* Complete the backward pass in `input_gradients`.
* Note that this operation does not have parameters, so `param_gradients` returns an empty dictionary.

Hint: Backpropagate through ReLU first, then through multiplication. For `x * y = 0`, you can assume the gradient of ReLU is 0.


In [ ]:
#@title CustomizedMulReLU

class CustomizedMulReLU(CustomizedLayer):
  def __init__(self):
    pass

  def set_params(self, param_dict: Dict[str, np.ndarray]) -> None:
    pass

  def forward(self, inputs: List[np.ndarray]) -> np.ndarray:
    x, y = inputs
    return np.maximum(x * y, 0)

  def input_gradients(
    self,
    inputs: List[np.ndarray],
    outputs: np.ndarray,
    output_gradient: np.ndarray
  ) -> Union[np.ndarray, List[np.ndarray]]:
    #####################
    # Your code goes here
    #####################
    x, y = inputs
    grad = output_gradient * (x * y > 0)
    return [grad * y, grad * x]

  def param_gradients(
      self,
      inputs: List[np.ndarray],
      outputs: np.ndarray,
      output_gradient: np.ndarray
  ) -> Dict[str, np.ndarray]:
    return {}

Run the test below. Do not change it.

In [7]:
#@title Run test

def verify_mul_relu():
  for width, height in [(3, 3), (10, 1), (100, 300)]:
    print(f'=== Testing with width={width} and height={height} ===')
    if width == 1:
      x = np.random.randn(height)
      y = np.random.randn(height)
    else:
      x = np.random.randn(height, width)
      y = np.random.randn(height, width)
    output_gradient = np.random.randn(*x.shape)
    customized_mul_relu = CustomizedMulReLU()
    customized_mul_relu.set_params({})
    class TorchMulReLU(nn.Module):
      def forward(self, x, y):
        return torch.relu(x * y)
    torch_mul_relu = TorchMulReLU()

    compare_with_actual_layer(
        torch_mul_relu,
        customized_mul_relu,
        [x, y],
        output_gradient,
        param_np=[],
        parameter_names=[],
    )

verify_mul_relu()

=== Testing with width=3 and height=3 ===
Output is close to the reference output.
Input Gradient 0 is close to the reference output.
Input Gradient 1 is close to the reference output.
All tests passed
=== Testing with width=10 and height=1 ===
Output is close to the reference output.
Input Gradient 0 is close to the reference output.
Input Gradient 1 is close to the reference output.
All tests passed
=== Testing with width=100 and height=300 ===
Output is close to the reference output.
Input Gradient 0 is close to the reference output.
Input Gradient 1 is close to the reference output.
All tests passed


## Question A.3 Back-propagation of Linear + ReLU


Similarly, complete the implementation of the class below, then run the test below.

The forward operation is `ReLU(x @ weight.T + bias)`.

* This time, we also ask you to implement the forward pass.
* Complete the backward pass in `input_gradients` and `param_gradients`.

Hint: Backpropagate through ReLU before the linear layer. You can assume the gradient of ReLU at 0 is 0.


In [ ]:
#@title CustomizedLinearReLU

class CustomizedLinearReLU(CustomizedLayer):
  def __init__(self, in_features, out_features):
    self.in_features = in_features
    self.out_features = out_features
    self.weight = np.zeros((out_features, in_features))
    self.bias = np.zeros(out_features)

  def set_params(self, param_dict: Dict[str, np.ndarray]):
    self.weight = param_dict['weight']
    self.bias = param_dict['bias']

  def forward(self, inputs: List[np.ndarray]) -> np.ndarray:
    #####################
    # Your code goes here
    #####################
    x = inputs[0]
    return np.maximum(x @ self.weight.T + self.bias, 0)

  def input_gradients(
    self,
    inputs: List[np.ndarray],
    outputs: np.ndarray,
    output_gradient: np.ndarray
  ) -> Union[np.ndarray, List[np.ndarray]]:
    #####################
    # Your code goes here
    #####################
    grad = output_gradient * (outputs > 0)
    return [grad @ self.weight]

  def param_gradients(
      self,
      inputs: List[np.ndarray],
      outputs: np.ndarray,
      output_gradient: np.ndarray
  ) -> Dict[str, np.ndarray]:
    #####################
    # Your code goes here
    #####################
    x = inputs[0]
    grad = output_gradient * (outputs > 0)
    return {'weight': grad.T @ x, 'bias': grad.sum(axis=0)}

Run the test below. Do not change it.

In [9]:
#@title Run test

def verify_linear_relu():
  for batch_size, in_features, out_features in [(3, 4, 8), (5, 10, 20), (2, 100, 300)]:
    print(f'=== Testing with in_features={in_features} '
          f'and out_features={out_features} ===')
    input_np = np.random.randn(batch_size, in_features)
    output_gradient = np.random.randn(batch_size, out_features)
    weight = np.random.randn(out_features, in_features)
    bias = np.random.randn(out_features)
    customized_linear_relu = CustomizedLinearReLU(in_features, out_features)
    customized_linear_relu.set_params({'weight': weight, 'bias': bias})
    class TorchLinearReLU(nn.Linear):
      def forward(self, x):
        return torch.relu(super().forward(x))
    torch_linear_relu = TorchLinearReLU(in_features, out_features)

    compare_with_actual_layer(
        torch_linear_relu,
        customized_linear_relu,
        [input_np],
        output_gradient,
        param_np=[weight, bias],
        parameter_names=['weight', 'bias'],
    )

verify_linear_relu()

=== Testing with in_features=4 and out_features=8 ===
Output is close to the reference output.
Input Gradient 0 is close to the reference output.
Parameter Gradient weight is close to the reference output.
Parameter Gradient bias is close to the reference output.
All tests passed
=== Testing with in_features=10 and out_features=20 ===
Output is close to the reference output.
Input Gradient 0 is close to the reference output.
Parameter Gradient weight is close to the reference output.
Parameter Gradient bias is close to the reference output.
All tests passed
=== Testing with in_features=100 and out_features=300 ===
Output is close to the reference output.
Input Gradient 0 is close to the reference output.
Parameter Gradient weight is close to the reference output.
Parameter Gradient bias is close to the reference output.
All tests passed


## Question A.4 Back-propagation of Conv

Finally, let us try a convolutional layer. To make your life easier, let us assume:

* Padding is always 1.
* Stride is always 1.
* The kernel is always square with an odd side length.

Similarly, complete the implementation of the class below, then run the test below.

* For your reference, we have provided the forward pass.
* Please complete the backward pass in `input_gradients` and `param_gradients`.

There is an additional requirement for this question:

* You cannot use any functions from NumPy, PyTorch, or other libraries that perform convolution.

Hint: Compute gradients using the zero-padded input, then remove the padded border from the input gradient before returning it.


In [ ]:
#@title CustomizedConv2d

class CustomizedConv2d(CustomizedLayer):
  def __init__(self, in_channels, out_channels, kernel_size):
    if (kernel_size % 2 == 0):
      raise ValueError(f"Kernel size {kernel_size} must be odd")
    self.in_channels = in_channels
    self.out_channels = out_channels
    self.kernel_size = kernel_size
    self.weight = np.zeros((out_channels, in_channels, kernel_size, kernel_size))
    self.bias = np.zeros(out_channels)

  def set_params(self, param_dict: Dict[str, np.ndarray]):
    self.weight = param_dict['weight']
    self.bias = param_dict['bias']

  def forward(self, inputs: List[np.ndarray]) -> np.ndarray:
    input = inputs[0] # [batch_size, in_channels, height, width]
    batch_size, in_channels, height, width = input.shape
    if self.in_channels != in_channels:
      raise ValueError(f'Input channels {in_channels} do not '
                       f'match expected {self.in_channels}')
    w = self.weight # [out_channels, in_channels, kernel_size, kernel_size]
    b = self.bias
    padded_input = np.pad(input, ((0, 0), (0, 0), (1, 1), (1, 1)),
                          mode='constant')
    out_height = height + 2 - self.kernel_size + 1
    out_width = width + 2 - self.kernel_size + 1
    output = np.zeros((batch_size, self.out_channels, out_height, out_width),
                      np.float64)
    for dx in range(0, self.kernel_size):
      for dy in range(0, self.kernel_size):
        # channel order: [B, out_channels, in_channels, H, W]
        y_start = dy
        y_end = y_start + out_height
        x_start = dx
        x_end = x_start + out_width
        output += (padded_input[:, np.newaxis, :, y_start:y_end, x_start:x_end] *
                   w[np.newaxis, :, :, np.newaxis, np.newaxis, dy, dx]
        ).sum(axis=2)
    output += b[np.newaxis, :, np.newaxis, np.newaxis]
    return output

  def input_gradients(
    self,
    inputs: List[np.ndarray],
    outputs: np.ndarray,
    output_gradient: np.ndarray
  ) -> Union[np.ndarray, List[np.ndarray]]:
    #####################
    # Your code goes here
    #####################
    x = inputs[0]
    batch_size, in_channels, height, width = x.shape
    out_height, out_width = output_gradient.shape[2:]
    padded_grad = np.zeros((batch_size, in_channels, height + 2, width + 2))

    for dy in range(self.kernel_size):
      for dx in range(self.kernel_size):
        w = self.weight[:, :, dy, dx]
        grad = (output_gradient[:, :, np.newaxis, :, :] * w[np.newaxis, :, :, np.newaxis, np.newaxis]).sum(axis=1)
        padded_grad[:, :, dy:dy + out_height, dx:dx + out_width] += grad

    return [padded_grad[:, :, 1:-1, 1:-1]]

  def param_gradients(
    self,
    inputs: List[np.ndarray],
    outputs: np.ndarray,
    output_gradient: np.ndarray
  ) -> Dict[str, np.ndarray]:
    #####################
    # Your code goes here
    #####################
    x = inputs[0]
    padded_x = np.pad(x, ((0, 0), (0, 0), (1, 1), (1, 1)))
    out_height, out_width = output_gradient.shape[2:]
    weight_grad = np.zeros_like(self.weight)

    for dy in range(self.kernel_size):
      for dx in range(self.kernel_size):
        patch = padded_x[:, :, dy:dy + out_height, dx:dx + out_width]
        weight_grad[:, :, dy, dx] = (
            output_gradient[:, :, np.newaxis, :, :] *
            patch[:, np.newaxis, :, :, :]
        ).sum(axis=(0, 3, 4))

    bias_grad = output_gradient.sum(axis=(0, 2, 3))
    return {'weight': weight_grad, 'bias': bias_grad}

Run the test below. Do not change it.

In [11]:
#@title Run test

def verify_conv2d():
  for batch_size, in_channels, out_channels, kernel_size, height, width in [
      (1, 1, 1, 3, 5, 5),
      (1, 4, 5, 3, 11, 11),
      (1, 4, 7, 5, 12, 12),
      (3, 4, 5, 3, 20, 30),
      (1, 5, 7, 5, 256, 256),
  ]:
    print(f'=== Testing with in_channels={in_channels} '
          f'and out_channels={out_channels} and kernel_size={kernel_size} ===')
    out_height = height + 2 - kernel_size + 1
    out_width = width + 2 - kernel_size + 1
    input_np = np.random.randn(batch_size, in_channels, height, width)
    output_gradient = np.random.randn(batch_size, out_channels, out_height, out_width)
    weight = np.random.randn(out_channels, in_channels, kernel_size, kernel_size)
    bias = np.random.randn(out_channels)
    customized_conv2d = CustomizedConv2d(in_channels, out_channels, kernel_size)
    customized_conv2d.set_params({'weight': weight, 'bias': bias})
    torch_conv2d = nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=1)

    compare_with_actual_layer(
        torch_conv2d,
        customized_conv2d,
        [input_np],
        output_gradient,
        param_np=[weight, bias],
        parameter_names=['weight', 'bias'],
    )

verify_conv2d()

=== Testing with in_channels=1 and out_channels=1 and kernel_size=3 ===
Output is close to the reference output.
Input Gradient 0 is close to the reference output.
Parameter Gradient weight is close to the reference output.
Parameter Gradient bias is close to the reference output.
All tests passed
=== Testing with in_channels=4 and out_channels=5 and kernel_size=3 ===
Output is close to the reference output.
Input Gradient 0 is close to the reference output.
Parameter Gradient weight is close to the reference output.
Parameter Gradient bias is close to the reference output.
All tests passed
=== Testing with in_channels=4 and out_channels=7 and kernel_size=5 ===
Output is close to the reference output.
Input Gradient 0 is close to the reference output.
Parameter Gradient weight is close to the reference output.
Parameter Gradient bias is close to the reference output.
All tests passed
=== Testing with in_channels=4 and out_channels=5 and kernel_size=3 ===
Output is close to the referenc

Output is close to the reference output.
Input Gradient 0 is close to the reference output.
Parameter Gradient weight is close to the reference output.
Parameter Gradient bias is close to the reference output.
All tests passed


Problem B. ResNet
==

To simplify network design, we resize the images to 32x32.

In [12]:
#@title Load the training dataset

# Define transformations for the training and test sets

mean_train = 0.2860405743122101
std_train = 0.3530242443084717
transform_train = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((mean_train,), (std_train,))
])

# Apply the same transformation to the training and test sets.
transform_test = transforms.Compose([
    # To simplify network design, we resize the images to 32x32.
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((mean_train,), (std_train,))
])

# Load the FashionMNIST dataset
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform_train
)

test_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform_test
)

Here are the same utility functions as in Assignment 2. If necessary, feel free to change them.

In [13]:
#@title Utility functions for training

def train_one_epoch(
    dataloader: torch.utils.data.DataLoader,
    model: nn.Module,
    loss_fn: nn.Module,
    optimizer: optim.Optimizer,
    device: str='cpu',
    loss_print_iter: int=100
  ):
  num_train_samples = len(dataloader.dataset)

  # Set the model to training mode.
  model.train()
  all_losses = []
  all_acc = []

  for batch_index, (image, label) in enumerate(dataloader):
    image, label = image.to(device), label.to(device)
    pred = model(image)
    loss = loss_fn(pred, label)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    # Calculate the loss
    all_losses.append(loss.item())
    acc = ((pred.argmax(1) == label).type(torch.float).sum().item() /
           image.shape[0])
    all_acc.append(acc)

    if batch_index % loss_print_iter == 0:
      loss, trained_samples = loss.item(), (batch_index + 1) * image.shape[0]
      print(f'loss: {loss:>7f} '
            f'[{trained_samples:>5d}/{num_train_samples:>5d}] ')

  return all_losses, all_acc

def test_all_samples(
    dataloader: torch.utils.data.DataLoader,
    model: nn.Module,
    loss_fn: nn.Module,
    device: str='cpu'
) -> Tuple[float, float]:
  model.eval()  # Set the model to evaluation mode
  num_testing_samples = len(dataloader.dataset)
  num_batches = len(dataloader)
  test_loss, correct = 0, 0

  # Disable gradient calculation for inference
  with torch.no_grad():
    for image, label in dataloader:
      image, label = image.to(device), label.to(device)
      pred = model(image)
      test_loss += loss_fn(pred, label).item()
      correct += (pred.argmax(1) == label).type(torch.float).sum().item()

  test_loss /= num_batches
  acc = correct / num_testing_samples
  print(f'Test Error: \n Accuracy: {(100*acc):>0.1f}%, '
        f'Avg loss: {test_loss:>8f} \n')
  return test_loss, acc

Next, let us select the device.

Using a GPU is recommended.

In [ ]:
if torch.cuda.is_available():
  device = torch.device("cuda")
elif torch.backends.mps.is_available():
  device = torch.device("mps")
else:
  device = torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


In this problem, let us verify why the ResNet structure is important for deep networks.

First, let us create the data loaders. Feel free to change the batch size.

In [15]:
# Create data loaders

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Example: A simple CNN.

No coding is required. Just run the following blocks.

In [16]:
#@title A utility function for creating the network

def gen_double_conv(in_channels, out_channels):
  return [nn.Conv2d(in_channels, out_channels, 3, padding=1),
          nn.ReLU(),
          nn.Conv2d(out_channels, out_channels, 3, padding=1),
          nn.ReLU()]

First, we will define a CNN with 11 convolution/linear layers and try it on FashionMNIST.

In [17]:
#@title SimpleCNN

class SimpleCNN(nn.Module):
  def __init__(
    self,
    img_size: int=32,
    in_channels: int=1,
    num_classes: int=10,
  ):
    super().__init__()
    base_channel = 16

    # Starting block. One 2x downsampling operation.
    self.start_conv = nn.Sequential(
        *gen_double_conv(in_channels, base_channel),
    )

    self.start_ds = nn.Sequential(
        nn.AvgPool2d(2),
    )

    # Mid block. No downsampling.
    self.mid_block = nn.Sequential(
        nn.Conv2d(base_channel, base_channel * 2, 3, padding=1),
        nn.ReLU(),
        nn.AvgPool2d(2),
        nn.Conv2d(base_channel * 2, base_channel * 2, 3, padding=1),
        nn.ReLU(),
        # Use ConvTranspose2d to increase the spatial resolution by a factor of 2.
        nn.ConvTranspose2d(base_channel * 2, base_channel, 2, stride=2),
        nn.ReLU(),
    )

    # Last block. Two 2x downsampling operations.
    self.end_conv1 = nn.Sequential(
        *gen_double_conv(base_channel, base_channel * 2)
    )
    self.end_ds1 = nn.Sequential(
        nn.AvgPool2d(2),
    )
    self.end_conv2 = nn.Sequential(
        *gen_double_conv(base_channel * 2, base_channel * 4)
    )

    n_pooling_layer = 3
    last_layer_size = img_size // (2 ** n_pooling_layer)
    self.end_ds2_linear = nn.Sequential(
        nn.AvgPool2d(2),
        nn.Flatten(),
        nn.Linear(last_layer_size * last_layer_size *
                  base_channel * 4, base_channel * 8),
        nn.ReLU(),
        nn.Linear(base_channel * 8, num_classes)
    )

  def forward(self, x):
    after_start = self.start_ds(self.start_conv(x))
    after_mid = self.mid_block(after_start)
    output = self.end_ds2_linear(
        self.end_conv2(self.end_ds1(self.end_conv1(after_mid))))
    return output

  def get_n_conv_linear_layers(self) -> int:
    all_blocks = [self.start_conv, self.start_ds,
                  self.mid_block,
                  self.end_conv1, self.end_ds1,
                  self.end_conv2, self.end_ds2_linear]
    module_list = (nn.ConvTranspose2d, nn.Conv2d, nn.Linear)
    nlayer = 0
    for block in all_blocks:
      nlayer += len([mod for mod in block.modules()
                    if isinstance(mod, module_list)])
    return nlayer

In [18]:
#@title Create a model and its optimizer

model = SimpleCNN()
model.to(device)
loss_fn = nn.CrossEntropyLoss()

epochs = 20   # Number of training epochs
optimizer = torch.optim.Adam(
  model.parameters(),    # All trainable parameters.
  lr=1e-4                # Learning rate
)

print(f'The model contains {model.get_n_conv_linear_layers()} conv or '
      f'linear layers')

The model contains 11 conv or linear layers


Run the training code below.

After training, you should be able to achieve 89% accuracy on the test set.

In [19]:
#@title Run training

for t in range(epochs):
  print(f"Epoch {t+1}\n-------------------------------")
  start_time =time.time()
  _, _ = train_one_epoch(train_loader, model, loss_fn, optimizer, device=device)
  print(f'One epoch takes {time.time() - start_time}')
  _, _ = test_all_samples(test_loader, model, loss_fn, device=device)

test_all_samples(test_loader, model, loss_fn, device=device)

print("Training done!")

Epoch 1
-------------------------------


loss: 2.292041 [   32/60000] 


loss: 2.298976 [ 3232/60000] 


loss: 1.361628 [ 6432/60000] 


loss: 1.128845 [ 9632/60000] 


...

loss: 0.110361 [54432/60000] 


loss: 0.332877 [57632/60000] 


One epoch takes 24.931922912597656


Test Error: 
 Accuracy: 90.8%, Avg loss: 0.257335 



Test Error: 
 Accuracy: 90.8%, Avg loss: 0.257335 

Training done!


## Problem B.1. Deeper CNN

Now, please change the network we have provided by making multiple independent copies of `self.mid_block`. Keep the remaining modules, their names and channel settings unchanged. The network you have created should have at least 30 conv/linear layers.

In [ ]:
class DeeperCNN(nn.Module):
  def __init__(
    self,
    img_size: int=32,
    in_channels: int=1,
    num_classes: int=10,
  ):
    #####################
    # Your code goes here
    #####################
    super().__init__()
    base_channel = 16

    self.start_conv = nn.Sequential(
        *gen_double_conv(in_channels, base_channel),                          # 1x32x32  -> 16x32x32
    )

    self.start_ds = nn.Sequential(nn.AvgPool2d(2))                            # 16x32x32 -> 16x16x16

    self.mid_block = nn.ModuleList([
        nn.Sequential(
            nn.Conv2d(base_channel, base_channel * 2, 3, padding=1),          # 16x16x16 -> 32x16x16
            nn.ReLU(),                                                        # 32x16x16 -> 32x16x16
            nn.AvgPool2d(2),                                                  # 32x16x16 -> 32x8x8
            nn.Conv2d(base_channel * 2, base_channel * 2, 3, padding=1),      # 32x8x8   -> 32x8x8
            nn.ReLU(),                                                        # 32x8x8   -> 32x8x8
            nn.ConvTranspose2d(base_channel * 2, base_channel, 2, stride=2),  # 32x8x8   -> 16x16x16
            nn.ReLU(),
        ) for _ in range(8)
    ])

    self.end_conv1 = nn.Sequential(
        *gen_double_conv(base_channel, base_channel * 2)                      # 16x16x16 -> 32x16x16
    )

    self.end_ds1 = nn.Sequential(nn.AvgPool2d(2))                             # 32x16x16 -> 32x8x8

    self.end_conv2 = nn.Sequential(
        *gen_double_conv(base_channel * 2, base_channel * 4)                  # 32x8x8   -> 64x8x8
    )

    n_pooling_layer = 3
    last_layer_size = img_size // (2 ** n_pooling_layer)
    self.end_ds2_linear = nn.Sequential(
        nn.AvgPool2d(2),                                                      # 64x8x8   -> 64x4x4
        nn.Flatten(),                                                         # 64x4x4   -> 1024
        nn.Linear(last_layer_size * last_layer_size *
                  base_channel * 4, base_channel * 8),                        # 1024     -> 128
        nn.ReLU(),                                                            # 128      -> 128
        nn.Linear(base_channel * 8, num_classes)                              # 128      -> 10
    )

  def forward(self, x):
    #####################
    # Your code goes here
    #####################
    after_start = self.start_ds(self.start_conv(x))
    after_mid = after_start
    for block in self.mid_block:
      after_mid = block(after_mid)
    output = self.end_ds2_linear(
        self.end_conv2(self.end_ds1(self.end_conv1(after_mid))))
    return output

  def get_n_conv_linear_layers(self) -> int:
    #####################
    # Your code goes here
    #####################
    return sum(isinstance(mod, (nn.ConvTranspose2d, nn.Conv2d, nn.Linear))
               for mod in self.modules())

In [21]:
#@title Create a model and its optimizer

model = DeeperCNN()
model.to(device)
loss_fn = nn.CrossEntropyLoss()

epochs = 20   # Number of training epochs
optimizer = torch.optim.Adam(
  model.parameters(),    # All trainable parameters.
  lr=1e-4                # Learning rate
)

print(f'The model contains {model.get_n_conv_linear_layers()} conv or '
      f'linear layers')

The model contains 32 conv or linear layers


Run the training code below.

Interestingly, you will find that with a deeper network, the accuracy actually goes down.

In [22]:
#@title Run training

for t in range(epochs):
  print(f"Epoch {t+1}\n-------------------------------")
  start_time =time.time()
  _, _ = train_one_epoch(train_loader, model, loss_fn, optimizer, device=device)
  print(f'One epoch takes {time.time() - start_time}')
  _, _ = test_all_samples(test_loader, model, loss_fn, device=device)

test_all_samples(test_loader, model, loss_fn, device=device)

print("Training done!")

Epoch 1
-------------------------------
loss: 2.318420 [   32/60000] 


loss: 2.283867 [ 3232/60000] 


loss: 2.311431 [ 6432/60000] 


loss: 2.315685 [ 9632/60000] 


loss: 2.304106 [12832/60000] 


...

loss: 2.302414 [54432/60000] 


loss: 2.303343 [57632/60000] 


One epoch takes 42.94412899017334


Test Error: 
 Accuracy: 10.0%, Avg loss: 2.302594 



Test Error: 
 Accuracy: 10.0%, Avg loss: 2.302594 

Training done!


## Problem B.2. ResNet

Now, keep the general structure in B.1 almost unchanged, but add some skip connections (ResNet structure). Try to achieve 89% accuracy again.

Hint: To add a skip connection when calling a module in `forward`, instead of:
```
y = net1(x)
```
use:
```
y = net1(x) + x
```

The input and output of each middle block have the same shape, so each copy of `self.mid_block` in your DeeperCNN can use a skip connection. Try to find where else you can add this skip connection.

Use the single-channel FashionMNIST input (`in_channels=1`) in B.2 and B.3. As a special case, `self.start_conv(x) + x` is also valid: PyTorch broadcasts the single input channel across the 16 output channels.

In [ ]:
class ResCNN(nn.Module):
  def __init__(
    self,
    img_size: int=32,
    in_channels: int=1,
    num_classes: int=10,
  ):
    #####################
    # Your code goes here
    #####################
    super().__init__()
    base_channel = 16

    self.start_conv = nn.Sequential(
        *gen_double_conv(in_channels, base_channel),                          # 1x32x32  -> 16x32x32
    )

    self.start_ds = nn.Sequential(nn.AvgPool2d(2))                            # 16x32x32 -> 16x16x16

    self.mid_block = nn.ModuleList([
        nn.Sequential(
            nn.Conv2d(base_channel, base_channel * 2, 3, padding=1),          # 16x16x16 -> 32x16x16
            nn.ReLU(),                                                        # 32x16x16 -> 32x16x16
            nn.AvgPool2d(2),                                                  # 32x16x16 -> 32x8x8
            nn.Conv2d(base_channel * 2, base_channel * 2, 3, padding=1),      # 32x8x8   -> 32x8x8
            nn.ReLU(),                                                        # 32x8x8   -> 32x8x8
            nn.ConvTranspose2d(base_channel * 2, base_channel, 2, stride=2),  # 32x8x8   -> 16x16x16
            nn.ReLU(),
        ) for _ in range(8)
    ])

    self.end_conv1 = nn.Sequential(
        *gen_double_conv(base_channel, base_channel * 2)                      # 16x16x16 -> 32x16x16
    )

    self.end_ds1 = nn.Sequential(nn.AvgPool2d(2))                             # 32x16x16 -> 32x8x8

    self.end_conv2 = nn.Sequential(
        *gen_double_conv(base_channel * 2, base_channel * 4)                  # 32x8x8   -> 64x8x8
    )

    n_pooling_layer = 3
    last_layer_size = img_size // (2 ** n_pooling_layer)
    self.end_ds2_linear = nn.Sequential(
        nn.AvgPool2d(2),                                                      # 64x8x8   -> 64x4x4
        nn.Flatten(),                                                         # 64x4x4   -> 1024
        nn.Linear(last_layer_size * last_layer_size *
                  base_channel * 4, base_channel * 8),                        # 1024     -> 128
        nn.ReLU(),                                                            # 128      -> 128
        nn.Linear(base_channel * 8, num_classes)                              # 128      -> 10
    )

  def forward(self, x):
    #####################
    # Your code goes here
    #####################
    after_start = self.start_ds(self.start_conv(x) + x)
    after_mid = after_start
    for block in self.mid_block:
      after_mid = block(after_mid) + after_mid
    output = self.end_ds2_linear(
        self.end_conv2(self.end_ds1(self.end_conv1(after_mid))))
    return output

  def get_n_conv_linear_layers(self) -> int:
    #####################
    # Your code goes here
    #####################
    return sum(isinstance(mod, (nn.ConvTranspose2d, nn.Conv2d, nn.Linear))
               for mod in self.modules())

In [24]:
#@title Create a model and its optimizer

model = ResCNN()
model.to(device)
loss_fn = nn.CrossEntropyLoss()

epochs = 20   # Number of training epochs
optimizer = torch.optim.Adam(
  model.parameters(),    # All trainable parameters.
  lr=1e-4                # Learning rate
)

print(f'The model contains {model.get_n_conv_linear_layers()} conv or '
      f'linear layers')

The model contains 32 conv or linear layers


Run the training code below.

Now, with the ResNet design, you should reach 89% accuracy again.

In [25]:
#@title Run training

for t in range(epochs):
  print(f"Epoch {t+1}\n-------------------------------")
  start_time =time.time()
  _, _ = train_one_epoch(train_loader, model, loss_fn, optimizer, device=device)
  print(f'One epoch takes {time.time() - start_time}')
  _, _ = test_all_samples(test_loader, model, loss_fn, device=device)

test_all_samples(test_loader, model, loss_fn, device=device)

print("Training done!")

Epoch 1
-------------------------------


loss: 2.296765 [   32/60000] 


loss: 0.931875 [ 3232/60000] 


loss: 1.230693 [ 6432/60000] 


loss: 0.807296 [ 9632/60000] 


...

loss: 0.067058 [54432/60000] 


loss: 0.154373 [57632/60000] 


One epoch takes 58.308436155319214


Test Error: 
 Accuracy: 90.7%, Avg loss: 0.269940 



Test Error: 
 Accuracy: 90.7%, Avg loss: 0.269940 

Training done!


## Problem B.3. Residual connections with different dimensions


Now, keep the general structure in B.2 almost unchanged, but add a skip connection to `self.end_conv1`.

The class below inherits from your `ResCNN` in B.2.

Hint: `super().__init__(...)` already creates the layers from B.2, so you do not need to define them again. Run the B.2 class definition first. You do not need to run its training code.

In B.2, the middle blocks have matching input and output shapes. However, `self.end_conv1` changes the number of channels from 16 to 32. Please transform the shortcut input so that it can be added to the output of this block.

Move the final ReLU of this block so that it is applied after the addition. Then run the training code below.

Hint: Use a `1x1` convolution with `stride=1` to match the number of channels while keeping the spatial dimensions unchanged.


In [ ]:
class ProjectionResCNN(ResCNN):
  def __init__(
    self,
    img_size: int=32,
    in_channels: int=1,
    num_classes: int=10,
  ):
    super().__init__(img_size, in_channels, num_classes)
    #####################
    # Your code goes here
    #####################
    self.shortcut = nn.Conv2d(16, 32, 1, stride=1)    # 16x16x16 -> 32x16x16

    self.end_relu = self.end_conv1[-1]
    
    self.end_conv1 = self.end_conv1[:-1]

  def forward(self, x):
    #####################
    # Your code goes here
    #####################
    after_start = self.start_ds(self.start_conv(x) + x)
    after_mid = after_start
    for block in self.mid_block:
      after_mid = block(after_mid) + after_mid
    after_end1 = self.end_relu(
        self.end_conv1(after_mid) + self.shortcut(after_mid))
    output = self.end_ds2_linear(
        self.end_conv2(self.end_ds1(after_end1)))
    return output

  def get_n_conv_linear_layers(self) -> int:
    #####################
    # Your code goes here
    #####################
    return sum(isinstance(mod, (nn.ConvTranspose2d, nn.Conv2d, nn.Linear))
               for mod in self.modules())


In [27]:
#@title Create a model and its optimizer

model = ProjectionResCNN()
model.to(device)
loss_fn = nn.CrossEntropyLoss()

epochs = 20   # Number of training epochs
optimizer = torch.optim.Adam(
  model.parameters(),    # All trainable parameters.
  lr=1e-4                # Learning rate
)

print(f'The model contains {model.get_n_conv_linear_layers()} conv or '
      f'linear layers')

The model contains 33 conv or linear layers


Run the training code below.


In [28]:
#@title Run training

for t in range(epochs):
  print(f"Epoch {t+1}\n-------------------------------")
  start_time =time.time()
  _, _ = train_one_epoch(train_loader, model, loss_fn, optimizer, device=device)
  print(f'One epoch takes {time.time() - start_time}')
  _, _ = test_all_samples(test_loader, model, loss_fn, device=device)

test_all_samples(test_loader, model, loss_fn, device=device)

print("Training done!")

Epoch 1
-------------------------------


loss: 2.309994 [   32/60000] 


loss: 0.811989 [ 3232/60000] 


loss: 0.631085 [ 6432/60000] 


loss: 0.935692 [ 9632/60000] 


...

loss: 0.127427 [54432/60000] 


loss: 0.065583 [57632/60000] 


One epoch takes 57.63493204116821


Test Error: 
 Accuracy: 90.5%, Avg loss: 0.278169 



Test Error: 
 Accuracy: 90.5%, Avg loss: 0.278169 

Training done!
